# Sneaker Resale Business Analysis: Defile de Mode (2022–2024)

**Author:** Baihaqsani  
**Tools:** Python, Pandas, Matplotlib  
**Data:** 1,038 transaction records from a personal sneaker resale business across 3 years

---

## Business Context

Defile de Mode is a sneaker resale business operating from 2019 to present, selling across multiple marketplace platforms including Alias, Kick Avenue, Direct Transfer, Tokopedia, Shopee, Ox Street, and Novelship. Capital is sourced from both personal funds and external investors, with profits shared accordingly.

## Questions This Analysis Answers

1. **Which sales channel is most capital-efficient?** (margin per channel)
2. **Which brand generates the best return on capital?** (margin per brand)
3. **How much capital is locked in unsold inventory?** (dead stock analysis)

## Key Findings (Summary)

- **Total net profit (2022–2024):** Rp 364,278,382 from 879 sold transactions
- **Alias outperforms Kick Avenue** by 6.5 margin points at comparable volume (~300 transactions each)
- **Adidas delivers 35% margin** vs Nike's 23%, despite Nike having 5x more transactions
- **Rp 288.7 million (16.5% of total capital) is locked in 159 unsold items**, all stagnant for over 1 year

---

## 1. Data Loading & Standardization

Transaction data is stored in three separate CSV files (one per year). Each file has a different header row position due to inconsistent formatting in the original spreadsheets. Column names also differ across years and were standardized before merging.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load each year — header row differs per file (verified manually)
df_2022 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2022.csv", header=4)
df_2023 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2023.csv", header=3)
df_2024 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2024.csv", header=3)

# Standardize column names across years
df_2022 = df_2022.rename(columns={"Nama Investor": "Investor"})
df_2024 = df_2024.rename(columns={"Harga Penjualan": "Penjualan"})

# Merge into single DataFrame
df_all = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

print(f"Total rows: {len(df_all)}")
df_all.info()

## 2. Data Cleaning

Four issues were identified and resolved:
1. **Money columns stored as text** — commas in numbers prevented numeric conversion
2. **Inconsistent channel names** — typos and spacing variations across entries
3. **Inconsistent brand names** — capitalization issues and one typo ("Addidas")
4. **21 zero-modal rows** — items won from raffles, recorded with no purchase cost

In [ ]:
# Convert money columns from text to numeric
# Commas used as thousand separators must be removed before conversion
# errors='coerce' turns unparseable values (e.g. '-') into NaN
money_cols = ["Modal", "Penjualan", "Pembagian Untung", "Untung Bersih"]

for col in money_cols:
    df_all[col] = pd.to_numeric(df_all[col].astype(str).str.replace(",", ""), errors="coerce")

# Clean channel names: fix spacing, capitalization, and typos
df_all["Platform Penjualan"] = df_all["Platform Penjualan"].str.strip().str.title()
df_all["Platform Penjualan"] = df_all["Platform Penjualan"].replace({
    "Direct Teansfer": "Direct Transfer",
    "Kick Avneue": "Kick Avenue",
    "Oxstreet": "Ox Street"
})

# Clean brand names: fix spacing, capitalization, and typos
df_all["Brand"] = df_all["Brand"].str.strip().str.title()
df_all["Brand"] = df_all["Brand"].replace({"Addidas": "Adidas"})

# Set display format for readability (does not alter data)
pd.options.display.float_format = '{:,.0f}'.format

print("Channels after cleaning:")
print(df_all["Platform Penjualan"].value_counts())
print("\nBrands after cleaning:")
print(df_all["Brand"].value_counts())

In [ ]:
# Split into sold vs unsold inventory
# Empty 'Penjualan' = item not yet sold (dead stock)
terjual = df_all[df_all["Penjualan"].notna()]    # sold items
dead_stock = df_all[df_all["Penjualan"].isna()]  # unsold items

# Separate zero-modal items (raffle winnings) for clean margin analysis
gratis = terjual[terjual["Modal"] == 0]      # 21 raffle items
bermodal = terjual[terjual["Modal"] > 0]     # 858 items with capital investment

# Verification
print(f"Sold: {len(terjual)} | Unsold (dead stock): {len(dead_stock)} | Total: {len(terjual) + len(dead_stock)}")
print(f"With capital: {len(bermodal)} | Raffle (zero cost): {len(gratis)}")

**Note on zero-modal items:** 21 items were won from sneaker raffles at no purchase cost. These are excluded from margin calculations (margin = profit ÷ capital — undefined when capital = 0) but reported separately as they represent pure profit with no capital at risk.

## 3. Overall Profit Summary

In [ ]:
total_profit = terjual["Untung Bersih"].sum()
raffle_profit = gratis["Untung Bersih"].sum()
capital_profit = bermodal["Untung Bersih"].sum()

print(f"Total net profit (all transactions):  Rp {total_profit:,.0f}")
print(f"  - From capital-backed items:        Rp {capital_profit:,.0f}")
print(f"  - From raffle items (zero cost):    Rp {raffle_profit:,.0f} ({raffle_profit/total_profit*100:.1f}% of total)")

Raffle items contributed only 0.9% of total profit across 21 transactions. While these items carry no capital risk (100% margin by definition), their absolute profit per transaction (Rp 162,723) is **2.6x lower** than capital-backed items (Rp 421,226), indicating that deliberately selected purchases consistently outperform random raffle wins in absolute returns.

## 4. Sales Channel Analysis

Margin is calculated as: **(total profit ÷ total capital invested) × 100**

Only items with capital > 0 are included to ensure margin reflects actual capital efficiency.

In [ ]:
channel = bermodal.groupby("Platform Penjualan").agg(
    total_modal=("Modal", "sum"),
    total_untung=("Untung Bersih", "sum")
)
channel["margin_pct"] = channel["total_untung"] / channel["total_modal"] * 100
channel["n_transactions"] = bermodal.groupby("Platform Penjualan")["Untung Bersih"].count()
channel.sort_values("margin_pct", ascending=False)

In [ ]:
channel_plot = channel.sort_values("margin_pct", ascending=True)

warna_channel = ["#4C72B0" if c != channel_plot["margin_pct"].idxmax()
                 else "#2CA02C" for c in channel_plot.index]

plt.figure(figsize=(9, 6))
bars = plt.barh(channel_plot.index, channel_plot["margin_pct"], color=warna_channel)

for bar, (idx, row) in zip(bars, channel_plot.iterrows()):
    lebar = bar.get_width()
    tengah_y = bar.get_y() + bar.get_height() / 2
    plt.text(lebar + 0.5, tengah_y, f"{lebar:.1f}%", va="center", fontsize=10)
    if lebar > 8:
        plt.text(lebar / 2, tengah_y, f"n={int(row['n_transactions'])}",
                 va="center", ha="center", fontsize=9, color="white")

plt.title("Profit Margin by Sales Channel", fontsize=13)
plt.xlabel("Margin (%)")
plt.ylabel("Channel")
plt.xlim(0, channel_plot["margin_pct"].max() + 8)
plt.tight_layout()
plt.show()

**Findings:**

Novelship (54.2%) and Ox Street (49.9%) show the highest margins, but with only 2 and 5 transactions respectively, these figures are statistically unreliable — a single outlier transaction can skew a small sample dramatically.

The most actionable comparison is **Alias vs Kick Avenue** — both have comparable transaction volumes (~300 each), making this a fair apples-to-apples comparison:
- **Alias: 28.5% margin** (n=301)
- **Kick Avenue: 22.0% margin** (n=307)

**Recommendation:** Shifting a portion of capital allocation from Kick Avenue to Alias has the potential to improve overall capital efficiency by approximately 6.5 percentage points without sacrificing transaction volume.

## 5. Brand Analysis

Same methodology as channel analysis. Zero-modal items are excluded.

In [ ]:
margin_brand_bersih = bermodal.groupby("Brand").agg(
    total_modal=("Modal", "sum"),
    total_untung=("Untung Bersih", "sum")
)
margin_brand_bersih["margin_pct"] = (
    margin_brand_bersih["total_untung"] / margin_brand_bersih["total_modal"] * 100
)
margin_brand_bersih["n_transactions"] = bermodal.groupby("Brand")["Untung Bersih"].count()
margin_brand_bersih.sort_values("margin_pct", ascending=False)

In [ ]:
brand_plot = margin_brand_bersih.copy()
brand_plot = brand_plot.sort_values("margin_pct", ascending=True)

warna_brand = ["#4C72B0" if c != brand_plot["margin_pct"].idxmax()
               else "#2CA02C" for c in brand_plot.index]

plt.figure(figsize=(9, 6))
bars = plt.barh(brand_plot.index, brand_plot["margin_pct"], color=warna_brand)

for bar, (idx, row) in zip(bars, brand_plot.iterrows()):
    lebar = bar.get_width()
    tengah_y = bar.get_y() + bar.get_height() / 2
    plt.text(lebar + 0.5, tengah_y, f"{lebar:.1f}%", va="center", fontsize=10)
    if lebar > 5:
        plt.text(lebar / 2, tengah_y, f"n={int(row['n_transactions'])}",
                 va="center", ha="center", fontsize=9, color="white")

plt.title("Profit Margin by Brand", fontsize=13)
plt.xlabel("Margin (%)")
plt.ylabel("Brand")
plt.xlim(0, brand_plot["margin_pct"].max() + 8)
plt.tight_layout()
plt.show()

**Findings:**

Converse (44.2%, n=4) and Crocs (36.5%, n=4) appear at the top, but with only 4 transactions each, these margins are not reliable for strategic decisions.

The statistically significant comparison is **Adidas vs Nike** — both have substantial transaction volumes:
- **Adidas: 35.1% margin** (n=123)
- **Nike: 22.8% margin** (n=662)

Despite Nike representing the majority of transactions (75% of volume), **Adidas generates 12 percentage points more margin per rupiah invested.** High volume does not equal high efficiency.

**Recommendation:** Increasing the proportion of Adidas purchases relative to Nike would improve overall capital efficiency. However, this must be balanced against market availability and liquidity — Nike's higher volume may partly reflect faster sell-through rates, which also affects capital turnover.

## 6. Dead Stock Analysis

Dead stock = items purchased but not yet sold. Capital invested in these items is effectively frozen and unavailable for reinvestment.

In [ ]:
# Capital locked in unsold inventory
modal_tertahan = dead_stock["Modal"].sum()
total_modal_semua = df_all["Modal"].sum()
persen_modal = modal_tertahan / total_modal_semua * 100
persen_unit = len(dead_stock) / len(df_all) * 100

print(f"Total capital invested (all items):  Rp {total_modal_semua:,.0f}")
print(f"Capital locked in dead stock:        Rp {modal_tertahan:,.0f} ({persen_modal:.1f}% of total capital)")
print(f"Units locked in dead stock:          {len(dead_stock)} units ({persen_unit:.1f}% of total units)")
print(f"\nCapital % ({persen_modal:.1f}%) > Unit % ({persen_unit:.1f}%) — unsold items skew slightly more expensive than average")

In [ ]:
# Break down dead stock by purchase year
dead_stock = dead_stock.copy()
dead_stock["tahun_beli"] = dead_stock["MM/DD/YY (Beli)"].str[-4:]
dead_stock["tahun_beli"] = dead_stock["tahun_beli"].replace({"0222": "2022"})  # fix data entry typo

ringkasan_tahun = dead_stock.groupby("tahun_beli").agg(
    jumlah_unit=("Modal", "size"),
    total_modal=("Modal", "sum")
)
ringkasan_tahun

In [ ]:
warna = ["#4C72B0", "#4C72B0", "#C44E52", "#4C72B0"]  # highlight 2024 (largest concentration)

plt.figure(figsize=(8, 5))
bar = plt.bar(ringkasan_tahun.index, ringkasan_tahun["total_modal"], color=warna)

for b in bar:
    tinggi = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, tinggi,
             f"Rp {tinggi/1_000_000:.0f}M",
             ha="center", va="bottom", fontsize=10)

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1_000_000:.0f}M")
)

plt.title("Capital Locked in Dead Stock by Purchase Year", fontsize=13)
plt.xlabel("Purchase Year")
plt.ylabel("Total Capital (Rp)")
plt.tight_layout()
plt.show()

**Findings:**

As of mid-2026, **all 159 unsold items have been stagnant for at least 1 year** — none qualify as "fresh stock." The breakdown:

| Year Purchased | Units | Capital Locked | Age |
|---|---|---|---|
| 2022 | 46 | Rp 96.8M | ~3.5 years |
| 2023 | 20 | Rp 41.8M | ~2.5 years |
| 2024 | 92 | Rp 149.0M | ~1.5 years |
| 2025 | 1 | Rp 1.1M | ~1 year |

**The 2024 cohort is the most critical concern** — 92 items representing Rp 149M (52% of all locked capital) purchased in a single year that failed to sell. This signals a potential misjudgment in purchasing decisions that year and warrants a detailed review of which brands/channels those items were intended for.

**Recommendation:** Prioritize liquidating the 2022 cohort first (longest-held, highest opportunity cost) even at reduced margins. Capital freed from dead stock can be redeployed into higher-margin channels (Alias) and brands (Adidas).

---

## 7. Conclusions & Strategic Recommendations

| # | Finding | Recommendation |
|---|---|---|
| 1 | Alias margin (28.5%) outperforms Kick Avenue (22.0%) at equal volume | Shift capital allocation toward Alias |
| 2 | Adidas margin (35.1%) far exceeds Nike (22.8%) | Increase Adidas purchase proportion |
| 3 | Rp 288.7M (16.5% of capital) locked in dead stock, all >1 year old | Begin structured liquidation, prioritize 2022 cohort |
| 4 | 2024 purchasing decisions resulted in largest dead stock concentration (Rp 149M) | Review 2024 buying criteria before scaling |

**Limitations:**
- Novelship and Ox Street margin data is unreliable due to very low transaction counts (n=2 and n=5)
- No customer-level data available (marketplace model) — customer segmentation analysis is not possible
- Date fields had inconsistent entry quality; holding period analysis was not performed
- This analysis covers closed transactions only; market conditions and pricing trends are not incorporated